# 🍕 How Zomato Keeps 10 Million Users Hooked — The Python Behind the Product

A **Zomato Analytics Engine**
- Store and manage order & restaurant data
- Process 500 orders to surface business insights
- Read & write real data files (CSV, JSON)
- Handle bad data **without ever crashing**

Topic 1 · Variables & Data Types\
Topic 2 · Loops & Iteration\
Topic 3 · Functions & Reusability\
Topic 4 · File I/O & JSON Parsing\
Topic 5 · Exception Handling

In [ ]:
# ============================================================
#  ZOMATO DATASET GENERATOR — Run this first!
# ============================================================

import random
import json
import csv
from datetime import datetime, timedelta
from collections import defaultdict

random.seed(42)   # same seed = same data for everyone in the room

# ── 20 Realistic Restaurants ─────────────────────────────────
RESTAURANTS = [
    {"id":"R001","name":"Biryani Blues",       "cuisine":"Biryani",       "city":"Delhi",     "rating":4.3,"avg_delivery_mins":35,"min_order":200},
    {"id":"R002","name":"Pizza Republic",       "cuisine":"Pizza",         "city":"Mumbai",    "rating":4.1,"avg_delivery_mins":28,"min_order":299},
    {"id":"R003","name":"Dosa Dynasty",         "cuisine":"South Indian",  "city":"Bangalore", "rating":4.5,"avg_delivery_mins":22,"min_order":150},
    {"id":"R004","name":"Wok This Way",         "cuisine":"Chinese",       "city":"Mumbai",    "rating":3.9,"avg_delivery_mins":30,"min_order":250},
    {"id":"R005","name":"Burger Barn",          "cuisine":"Burgers",       "city":"Hyderabad", "rating":4.2,"avg_delivery_mins":20,"min_order":199},
    {"id":"R006","name":"Kebab Kingdom",        "cuisine":"North Indian",  "city":"Delhi",     "rating":4.4,"avg_delivery_mins":40,"min_order":300},
    {"id":"R007","name":"Taco Fiesta",          "cuisine":"Mexican",       "city":"Bangalore", "rating":3.8,"avg_delivery_mins":25,"min_order":249},
    {"id":"R008","name":"Sushi Sutra",          "cuisine":"Japanese",      "city":"Mumbai",    "rating":4.6,"avg_delivery_mins":45,"min_order":499},
    {"id":"R009","name":"Chole Bhature House",  "cuisine":"North Indian",  "city":"Delhi",     "rating":4.0,"avg_delivery_mins":30,"min_order":180},
    {"id":"R010","name":"The Dessert Lab",      "cuisine":"Desserts",      "city":"Chennai",   "rating":4.7,"avg_delivery_mins":18,"min_order":149},
    {"id":"R011","name":"Andhra Spice",         "cuisine":"South Indian",  "city":"Hyderabad", "rating":4.3,"avg_delivery_mins":27,"min_order":175},
    {"id":"R012","name":"Pav Bhaji Planet",     "cuisine":"Street Food",   "city":"Mumbai",    "rating":4.1,"avg_delivery_mins":20,"min_order":120},
    {"id":"R013","name":"Noodle Nirvana",       "cuisine":"Chinese",       "city":"Bangalore", "rating":3.7,"avg_delivery_mins":35,"min_order":220},
    {"id":"R014","name":"Rolls Royce",          "cuisine":"Rolls & Wraps", "city":"Delhi",     "rating":4.2,"avg_delivery_mins":22,"min_order":160},
    {"id":"R015","name":"Curry Leaf",           "cuisine":"South Indian",  "city":"Chennai",   "rating":4.4,"avg_delivery_mins":25,"min_order":200},
    {"id":"R016","name":"Shawarma Station",     "cuisine":"Middle Eastern","city":"Hyderabad", "rating":4.0,"avg_delivery_mins":20,"min_order":149},
    {"id":"R017","name":"The Pancake House",    "cuisine":"Continental",   "city":"Bangalore", "rating":4.5,"avg_delivery_mins":30,"min_order":299},
    {"id":"R018","name":"Rajma Chawal Corner",  "cuisine":"North Indian",  "city":"Delhi",     "rating":3.9,"avg_delivery_mins":28,"min_order":140},
    {"id":"R019","name":"Smoothie Republic",    "cuisine":"Beverages",     "city":"Chennai",   "rating":4.3,"avg_delivery_mins":15,"min_order":99},
    {"id":"R020","name":"Mumbai Tiffin Box",    "cuisine":"Home Food",     "city":"Mumbai",    "rating":4.6,"avg_delivery_mins":38,"min_order":175},
]

PAYMENT_MODES = ["UPI","Credit Card","Debit Card","Wallet","Cash on Delivery"]
STATUSES      = ["Delivered","Delivered","Delivered","Cancelled","In Progress"]  # weighted toward Delivered
base_date     = datetime(2024, 1, 1)

# ── Generate 500 Orders ───────────────────────────────────────
orders = []
for i in range(500):
    r        = random.choice(RESTAURANTS)
    qty      = random.randint(1, 5)
    u_price  = random.randint(r["min_order"] // 2, r["min_order"] * 3)
    del_mins = r["avg_delivery_mins"] + random.randint(-10, 20)
    o_time   = base_date + timedelta(days=random.randint(0,89), hours=random.randint(9,23), minutes=random.randint(0,59))

    order = {
        "order_id"          : f"ZOM{str(i+1).zfill(5)}",
        "user_id"           : f"USR{random.randint(1000,9999)}",
        "restaurant_id"     : r["id"],
        "restaurant_name"   : r["name"],
        "cuisine"           : r["cuisine"],
        "city"              : r["city"],
        "quantity"          : qty,
        "unit_price"        : u_price,
        "total_amount"      : round(qty * u_price * random.uniform(0.85, 1.15), 2),
        "delivery_time_mins": max(10, del_mins),
        "rating"            : round(random.uniform(2.5, 5.0), 1) if random.random() > 0.2 else None,
        "payment_mode"      : random.choice(PAYMENT_MODES),
        "status"            : random.choice(STATUSES),
        "order_time"        : o_time.strftime("%Y-%m-%d %H:%M:%S"),
        "is_first_order"    : random.random() < 0.15,
    }

    if i in [47, 123, 287, 401]:
        order["total_amount"] = "MISSING"
    if i == 199:
        order["rating"] = "five stars"

    orders.append(order)

# ── Save to files ─────────────────────────────────────────────
with open("zomato_orders.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=orders[0].keys())
    writer.writeheader()
    writer.writerows(orders)

with open("restaurants.json", "w") as f:
    json.dump(RESTAURANTS, f, indent=2)

# ── Quick sanity check ────────────────────────────────────────
valid_amounts = [o["total_amount"] for o in orders if isinstance(o["total_amount"], float)]
print("✅  Dataset ready!")
print(f"    📄 zomato_orders.csv  → {len(orders)} orders")
print(f"    📋 restaurants.json   → {len(RESTAURANTS)} restaurants")
print(f"\n📊 Quick Snapshot:")
cities = list({o["city"] for o in orders})
print(f"    Cities   : {', '.join(sorted(cities))}")
print(f"    Revenue  : ₹{sum(valid_amounts):,.0f}")
print(f"    Delivered: {sum(1 for o in orders if o['status']=='Delivered')}")
print(f"    Cancelled: {sum(1 for o in orders if o['status']=='Cancelled')}")
print(f"    Dirty rows: {sum(1 for o in orders if o['total_amount']=='MISSING' or o['rating']=='five stars')}")

In [ ]:
import csv, json, random
from datetime import datetime
from collections import defaultdict, Counter

# ── Load orders ───────────────────────────────────────────────
with open("zomato_orders.csv", "r") as f:
    all_orders = list(csv.DictReader(f))

# ── Load restaurants ──────────────────────────────────────────
with open("restaurants.json", "r") as f:
    all_restaurants = json.load(f)

# ── Quick check ────────────────────────────────────────
print(f"✅  Orders loaded      : {len(all_orders)}")
print(f"✅  Restaurants loaded : {len(all_restaurants)}")
# print(f"\n📋 Order columns ({len(all_orders[0])}):")
# for col in all_orders[0].keys():
#     print(f"    • {col}")
# print(f"\n🔍 First order preview:")
# for k, v in all_orders[0].items():
#     print(f"    {k:<22} → {v}")

✅  Orders loaded      : 500
✅  Restaurants loaded : 20


Task 1: Inspect one order and label every data type. We did computations.



In [ ]:
one_order = all_orders[0]
print("Order Details")
print(one_order)
for key,value in one_order.items():
    dtype=type(value).__name__
    print(f"{key}: {type(value)}")

order_id    = one_order["order_id"]          # str
amount      = float(one_order["total_amount"])  # str from CSV → convert to float
qty         = int(one_order["quantity"])     # str from CSV → convert to int
status      = one_order["status"]            # str
first_order = one_order["is_first_order"]   # str "True"/"False" from CSV

print(f"\n💡 Computed:")
print(f"  Unit price inferred  : ₹{amount/qty:.2f}")
print(f"  Is a new customer?   : {first_order}")

payment_options = ["UPI", "Credit Card", "Debit Card", "Wallet", "Cash on Delivery"]
print(f"\n  Payment options      : {payment_options}")
print(f"  Default option       : {payment_options[0]}")   # index 0 = first item
print(f"  Total options        : {len(payment_options)}")

Order Details
{'order_id': 'ZOM00001', 'user_id': 'USR2679', 'restaurant_id': 'R004', 'restaurant_name': 'Wok This Way', 'cuisine': 'Chinese', 'city': 'Mumbai', 'quantity': '1', 'unit_price': '406', 'total_amount': '427.52', 'delivery_time_mins': '27', 'rating': '2.7', 'payment_mode': 'Wallet', 'status': 'Delivered', 'order_time': '2024-01-29 11:47:00', 'is_first_order': 'True'}
order_id: <class 'str'>
user_id: <class 'str'>
restaurant_id: <class 'str'>
restaurant_name: <class 'str'>
cuisine: <class 'str'>
city: <class 'str'>
quantity: <class 'str'>
unit_price: <class 'str'>
total_amount: <class 'str'>
delivery_time_mins: <class 'str'>
rating: <class 'str'>
payment_mode: <class 'str'>
status: <class 'str'>
order_time: <class 'str'>
is_first_order: <class 'str'>

💡 Computed:
  Unit price inferred  : ₹427.52
  Is a new customer?   : True

  Payment options      : ['UPI', 'Credit Card', 'Debit Card', 'Wallet', 'Cash on Delivery']
  Default option       : UPI
  Total options        : 5


11th order - Unit price and Is a new customer or not?

In [ ]:
order_11=all_orders[10]
for key, value in order_11.items():
    dtype=type(value).__name__
    print(f"{key}: {type(value)}")

amount=float(order_11["total_amount"])
qty=int(order_11["quantity"])
print(f"price per item: ₹{amount/qty:.2f}")
first_order11 = order_11["is_first_order"]   # str "True"/"False" from CSV
print(f"  Is a new customer?   : {first_order11}")

order_id: <class 'str'>
user_id: <class 'str'>
restaurant_id: <class 'str'>
restaurant_name: <class 'str'>
cuisine: <class 'str'>
city: <class 'str'>
quantity: <class 'str'>
unit_price: <class 'str'>
total_amount: <class 'str'>
delivery_time_mins: <class 'str'>
rating: <class 'str'>
payment_mode: <class 'str'>
status: <class 'str'>
order_time: <class 'str'>
is_first_order: <class 'str'>
price per item: ₹706.83
  Is a new customer?   : False


**Task 2:** A new restaurant has just signed up on Zomato. Create a `dict` called `new_restaurant` with at least 6 fields (name, cuisine, city, rating, avg_delivery_mins, min_order). Then create a `list` called `available_cuisines` with 5 cuisine names. Print both.


In [ ]:
# Hint: use curly braces {} for a dict, square brackets [] for a list
# Hint: make sure rating is a float (e.g. 4.2), not a string ("4.2")
# Hint: print each field of the dict on its own line using a for loop

# Your code here ↓

new_resturant =  {
    "name":"lucky",
    "cuisin":"chinese",
    "city":"hyd",
    "rating":4,
    "avg_delivery_min": "20min",
    "min_orders":3
}
available_cuisines = ["chinese","south_food","north_food","amma_cheti_vanta","cook_resturant"]

**Task:** From `all_orders`, pick ** 3 orders** (by index - 65, 43, 359). For each, print the `order_id`, `restaurant_name`, `total_amount` as a float, and `status`. Then check: which of the 3 has the highest amount?

In [ ]:
three_orders=[all_orders[65],all_orders[43],all_orders[359],all_orders[23],all_orders[47]]
for o in three_orders:
    try:
      amt=float(o["total_amount"])
    except ValueError:
        amt=0.0
    print(f"{o["order_id"]}{o["restaurant_name"]}{o["total_amount"]}{o["status"]}")

valid = [(o, float(o["total_amount"])) for o in three_orders
         if o["total_amount"] != "MISSING"]
best  = min(valid, key=lambda x: x[1])
print(f"\nBest order: {best[0]['order_id']} for ₹{best[1]:.2f}")

#valid (o1,348.25)

ZOM00066Curry Leaf348.25In Progress
ZOM00044Noodle Nirvana736.33In Progress
ZOM00360Dosa Dynasty1936.51Cancelled
ZOM00024Smoothie Republic75.71Delivered
ZOM00048Rajma Chawal CornerMISSINGIn Progress

Best order: ZOM00024 for ₹75.71


Loop through all 500 orders. Compute total revenue for Delhi + order count for Delhi

In [ ]:
target_city="Delhi"
delhi_revenue=0.0
delhi_orders=0

for order in all_orders:
  if order["city"]==target_city:
    try:
      delhi_revenue+=float(order["total_amount"])
      delhi_orders+=1
    except ValueError:
      pass

avg_order_delhi=delhi_revenue/delhi_orders if delhi_orders>0 else 0

print(f"Target city: {target_city}")
print(f"   Total orders  : {delhi_orders}")
print(f"   Total revenue : ₹{delhi_revenue:,.2f}")
print(f"   Avg order val : ₹{avg_order_delhi:,.2f}")

Target city: Delhi
   Total orders  : 111
   Total revenue : ₹94,919.54
   Avg order val : ₹855.13


**Task:** Count how many orders were placed for **each cuisine** across all 500 orders. Print the top 5 most-ordered cuisines with their order counts, ranked highest to lowest.


In [ ]:
cuisine_count={}

for order in all_orders:
  cuisine=order["cuisine"]
  if cuisine not in cuisine_count:
    cuisine_count[cuisine]=0
  cuisine_count[cuisine]+=1

top10=sorted(cuisine_count.items(),key=lambda x:x[1],reverse=True)[:10]

print("top 10 cuisines")

for rank, (cuisine,count) in enumerate(top10,1):
  print(f"{rank}. {cuisine} ({count} orders)")

top 10 cuisines
1. South Indian (82 orders)
2. North Indian (68 orders)
3. Chinese (60 orders)
4. Rolls & Wraps (31 orders)
5. Home Food (31 orders)
6. Burgers (31 orders)
7. Middle Eastern (26 orders)
8. Pizza (25 orders)
9. Japanese (24 orders)
10. Continental (24 orders)


In [ ]:
square=lambda x:x**2
print(square(10))

100
